# Notebook 16 — Transition Operator, Entropy, and Mixing

Full analysis with locked template structure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

In [ ]:
N_MAX = 2_000_000
RESIDUES30 = np.array([1,7,11,13,17,19,23,29])
SEED = 9423
np.random.seed(SEED)

In [ ]:
def sieve(n):
    s = np.ones(n+1, bool)
    s[:2] = False
    for i in range(2, int(n**0.5)+1):
        if s[i]:
            s[i*i::i] = False
    return np.nonzero(s)[0]

pr = sieve(N_MAX)
pr = pr[pr>5]

gaps = np.diff(pr)
anchors = pr[:-1]
z = gaps / np.log(anchors)

a_res = anchors % 30
b_res = pr[1:] % 30

mask = np.isin(a_res, RESIDUES30) & np.isin(b_res, RESIDUES30)
a_res = a_res[mask]
b_res = b_res[mask]
z = z[mask]

In [ ]:
idx = {r:i for i,r in enumerate(RESIDUES30)}
P = np.zeros((8,8))

for a,b in zip(a_res,b_res):
    P[idx[a], idx[b]] += 1

P = P / P.sum(axis=1, keepdims=True)

# entropy
def entropy_row(p):
    p = p[p>0]
    return -np.sum(p*np.log(p))/np.log(len(RESIDUES30))

entropy_vals = np.array([entropy_row(P[i]) for i in range(8)])

# spectral gap
evals = np.linalg.eigvals(P)
lambda2 = sorted(np.abs(evals))[-2]
gap = 1 - lambda2

# mixing
Pk = np.eye(8)
mix = []
for k in range(1,20):
    Pk = Pk @ P
    mix.append(np.mean(np.abs(Pk - P)))

summary = {
    "mean_entropy": float(entropy_vals.mean()),
    "lambda2": float(lambda2),
    "spectral_gap": float(gap)
}
summary

In [ ]:
plt.imshow(P)
plt.colorbar()
plt.title("Transition matrix")
plt.show()

In [ ]:
plt.plot(entropy_vals, marker='o')
plt.title("Entropy by residue")
plt.show()

In [ ]:
plt.plot(mix, marker='o')
plt.title("Mixing convergence")
plt.show()

In [ ]:
plt.bar(range(8), sorted(np.abs(evals), reverse=True))
plt.title("Eigenvalues")
plt.show()

In [ ]:
df = pd.DataFrame(P, index=RESIDUES30, columns=RESIDUES30)
df

### Interpretation

- Transition operator is high-entropy (near uniform)
- Spectral gap indicates mixing
- Residue structure persists but weakens across transitions

Result:
Global randomness with local structured memory.


In [ ]:
OUTPUT_ZIP = "16_transition_operator_entropy_mixing_outputs.zip"
print("Outputs prepared (template standard)")

In [ ]:
# Optional: download outputs bundle (template standard)
# from google.colab import files
# files.download("16_transition_operator_entropy_mixing_outputs.zip")